# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. The dataset schema is defined in Croissant format and is accessible using the provided URL.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print("Description:")
print(metadata.description)
print("\nPublished:", getattr(metadata, 'datePublished', 'unknown'))
print("\nFields:")
fields = dir(metadata)
pprint.pprint([f for f in fields if not f.startswith('__') and not callable(getattr(metadata, f))])

## 2. Data Overview
Review available record sets, their `@id` fields, and their data fields (all referenced by `@id`).

In [ ]:
# List all record sets (@id) available in the dataset

# Dataset may have a list named 'record_sets' (Croissant 1.0 convention)
if hasattr(dataset, "record_sets"):
    recsets = dataset.record_sets
else:
    recsets = [x for x in dir(dataset)
               if isinstance(getattr(dataset, x), mlc.data_structure.RecordSet)]

print("Available record sets (@id):")
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if isinstance(rs, dict) and '@id' in rs:
            print(f"- {rs['@id']}")
        elif isinstance(rs, str):
            print(f"- {rs}")
else:
    print("No record sets found in metadata.")

# Directly access via dataset API:
record_set_ids = []
if hasattr(dataset, 'record_sets'):
    for recset in dataset.record_sets:
        print(f"- {recset.id}")
        record_set_ids.append(recset.id)

    # Show fields for each record set (by @id)
    for recset in dataset.record_sets:
        print(f"\nRecord set: {recset.id}")
        for field in recset.fields:
            print(f"  Field @id: {field.id} (name: {getattr(field, 'name', '')})")
else:
    print("Could not find any record sets via Croissant API.")

## 3. Data Extraction
Load data from a specific record set as a DataFrame for analysis. All data is referenced by the record set and field `@id`s found above.

> **Tip:** Below, we attempt to load the first available record set in the list. Adjust the `record_set_ids` and field names if needed.

In [ ]:
# For demonstration, grab all available record set @ids
if 'record_set_ids' not in locals() or not record_set_ids:
    # fallback in case the metadata is not populated
    record_set_ids = []

# Load all available record sets as DataFrames using their @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(2), "\n---\n")

# For further analysis, select the first record set if available
if record_set_ids:
    first_set_id = record_set_ids[0]
    print(f"Using record set for analysis: {first_set_id}")
else:
    first_set_id = None

if first_set_id:
    print(f"Sample columns: {dataframes[first_set_id].columns.tolist()}")
    display(dataframes[first_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform typical data processing: filter records, normalize numeric fields, and group by a field. All fields referenced by their `@id`.

In [ ]:
# Example: Suppose we want to use the first numeric field found in this record set.
import numpy as np
# Identify the first numeric column (float or int) in the DataFrame
numeric_field_id = None
group_field_id = None
if first_set_id and not dataframes[first_set_id].empty:
    for col in dataframes[first_set_id].columns:
        col_dtype = dataframes[first_set_id][col].dropna().infer_objects().dtype
        if np.issubdtype(col_dtype, np.number):
            numeric_field_id = col
            break
    # For demonstration, pick next available column as group field
    for col in dataframes[first_set_id].columns:
        if col != numeric_field_id:
            group_field_id = col
            break

print(f"Numeric field (@id) for EDA: {numeric_field_id}")
print(f"Group field (@id) for grouping: {group_field_id}")

if numeric_field_id and first_set_id:
    threshold = dataframes[first_set_id][numeric_field_id].quantile(0.95) if not dataframes[first_set_id][numeric_field_id].isnull().all() else 10
    filtered_df = dataframes[first_set_id][dataframes[first_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mean_ = filtered_df[numeric_field_id].mean()
    std_ = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_ if std_ else filtered_df[numeric_field_id]
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (average {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("Insufficient data, or no numeric fields identified.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship with the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_set_id and numeric_field_id:
    df = dataframes[first_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated step-by-step how to use `mlcroissant` to:
- Load metadata and record sets from a Croissant dataset schema by URL
- Explore available record sets and fields (by `@id`)
- Extract tabular data for analysis
- Perform exploratory data analysis, including filtering and normalization
- Visualize numeric data distributions and categorical groupings

**All dataset elements are referenced by their `@id`, ensuring clarity and reproducibility across operations.**

The FAIR² package offers extensive metadata describing its collection, variables, and potential biases. For advanced processing—such as modeling or joining multiple record sets—review the official schema and refer to the relevant `@id`s for data extraction.

_Be sure to consult the full Croissant schema for variable definitions and further documentation._